# Base de Dados para Modelo de Recomendação - Aventura Teatros

## 1. Introdução

O Portal Aventura atualmente não possui um sistema estruturado de recomendação de espetáculos. Isso faz com que visitantes acessem uma página de um espetáculo sem receber sugestões de outras experiências relevantes, reduzindo a descoberta de conteúdo e a conversão em vendas.

Este notebook tem como objetivo estruturar e preparar a base de dados que alimentará o modelo de **filtragem colaborativa**. Nessa abordagem, as recomendações são geradas com base no padrão de avaliações de usuários com perfis semelhantes, ou seja, se dois usuários avaliaram positivamente espetáculos parecidos no passado, o sistema recomenda ao primeiro o que o segundo também gostou.

## 1.1 Estratégias de Formatação

Base inicial (colunas método de pagamento, canal da venda removidas)
Avaliacao (populada c maior peso em 3 e 4)
Tipo_producao (cruzou dados da tabela inicial com tabela nova que continha informacoes da produção)


## 2. Carregamento do Dataset

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo de visualização
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

# Carregamento
df = pd.read_csv('../database/basededados-spotlight_atualizado.csv')
print(f' Dataset carregado com sucesso!')
print(f'   Registros: {len(df):,}')
print(f'   Colunas:   {df.shape[1]}')
df.head()


 Dataset carregado com sucesso!
   Registros: 343,974
   Colunas:   10


,id_user,nome_evento,teatro,data_da_apresentacao,data_da_compra,setor,valor_do_ingresso,tipo_producao,categoria,avaliacao
0,1,um mallandro na quarentena com sergio mallandro,TEATRO RIACHUELO RJ,6/26/21 20:00,5/20/21 5:03,PLATEIA VIP - 02 PESSOAS,120,EXTERNA,Show Stand-Up,3
1,1,um mallandro na quarentena com sergio mallandro,TEATRO RIACHUELO RJ,6/26/21 20:00,5/20/21 5:03,PLATEIA VIP - 02 PESSOAS,120,EXTERNA,Show Stand-Up,4
2,2,dominguinhos - isso aqui tá bom demais,TEATRO RIACHUELO RJ,6/9/23 20:00,6/9/23 10:04,BALCÃO,50,EXTERNA,Show Musical,5
3,2,dominguinhos - isso aqui tá bom demais,TEATRO RIACHUELO RJ,6/9/23 20:00,6/9/23 10:04,BALCÃO,50,EXTERNA,Show Musical,3
4,3,a noviça rebelde,TEATRO RIACHUELO RJ,5/11/24 19:00,5/10/24 20:33,PLATEIA,225,AVENTURA,Espetáculo Musical,3


## 3. Descrição do Dataset

In [20]:
print('=== Informações Gerais do Dataset ===') 
print(f'Registros totais       : {len(df):>10,}')
print(f'Usuários únicos        : {df["id_user"].nunique():>10,}')
print(f'Espetáculos únicos     : {df["nome_evento"].nunique():>10,}')
print(f'Teatros únicos         : {df["teatro"].nunique():>10,}')
print(f'Categorias únicas      : {df["categoria"].nunique():>10,}')
print('\nColunas disponíveis:')
for col in df.columns:
    print(f'  - {col} ({df[col].dtype})')
print()
print('=== Tipos e Nulos ===')    
print(df.dtypes)
print()
print('Valores nulos por coluna:')
print(df.isnull().sum())


=== Informações Gerais do Dataset ===
Registros totais       :    343,974
Usuários únicos        :     68,564
Espetáculos únicos     :        571
Teatros únicos         :          4
Categorias únicas      :          8

Colunas disponíveis:
  - id_user (int64)
  - nome_evento (object)
  - teatro (object)
  - data_da_apresentacao (object)
  - data_da_compra (object)
  - setor (object)
  - valor_do_ingresso (object)
  - tipo_producao (object)
  - categoria (object)
  - avaliacao (int64)

=== Tipos e Nulos ===
id_user                  int64
nome_evento             object
teatro                  object
data_da_apresentacao    object
data_da_compra          object
setor                   object
valor_do_ingresso       object
tipo_producao           object
categoria               object
avaliacao                int64
dtype: object

Valores nulos por coluna:
id_user                 0
nome_evento             0
teatro                  0
data_da_apresentacao    1
data_da_compra          1
setor  

In [21]:
print('Estatísticas descritivas das avaliações:')
df[['avaliacao']].describe().round(2)

Estatísticas descritivas das avaliações:


,avaliacao
count,343974.00
mean,3.10
std,1.12
min,1.00
25%,2.00
50%,3.00
75%,4.00
max,5.00


In [22]:
print('Interações por usuário:')
user_counts = df.groupby('id_user').size()
display(user_counts.describe().rename('interações').to_frame())

print('\nInterações por espetáculo:')
item_counts = df.groupby('nome_evento').size()
display(item_counts.describe().rename('interações').to_frame())

Interações por usuário:


,interações
count,68564.000000
mean,5.016831
std,304.622645
min,1.000000
25%,2.000000
50%,2.000000
75%,4.000000
max,79721.000000



Interações por espetáculo:


,interações
count,571.000000
mean,602.406305
std,1916.956592
min,1.000000
25%,63.500000
50%,154.000000
75%,402.000000
max,32475.000000
